# Практическая работа №3: Работа с реальными данными

Цель работы: Научиться работать с реальными файлами данных разных форматов, читать большие CSV по частям, выбирать только нужные столбцы, задавать типы данных, импортировать JSON и Excel и объединять несколько файлов в один DataFrame.

## 1. Подготовка окружения и данных

### 1.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import glob

### 1.2. Описание датасетов

В работе используются «реальные» наборы данных, подготовленные в папке `shared/data`:

- Логи заказов интернет-магазина в формате CSV (несколько файлов за разные месяцы): `orders_2024_01.csv`, `orders_2024_02.csv`, `orders_2024_03.csv`.
- Дополнительная справочная информация в формате Excel (категории товаров): `products.xlsx`.
- Логи событий/кликов в формате JSON (форматы records и json lines): `events_records.json`, `events_lines.json`.

Структура заказов: `order_id`, `user_id`, `order_datetime`, `product_id`, `price`, `quantity`, `status` (`paid`, `cancelled`, `refunded`).

## 2. Чтение больших CSV по частям (chunksize)

### 2.1. Проблема больших CSV

Когда CSV-файл содержит миллионы строк, его полная загрузка в память может быть невозможна или очень медленной, поэтому используется построчная/пакетная обработка с помощью параметра `chunksize`.

### 2.2. Чтение большого CSV по частям и подсчёт выручки

In [2]:
csv_path = "../shared/data/orders_2024_01.csv"

chunk_size = 1_000  # количество строк в одном чанке
total_revenue = 0
rows_count = 0

for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    # вычисляем выручку в чанке
    chunk["revenue"] = chunk["price"] * chunk["quantity"]
    total_revenue += chunk["revenue"].sum()
    rows_count += len(chunk)

print("Общее количество строк:", rows_count)
print("Общая выручка за период:", total_revenue)

Общее количество строк: 5000
Общая выручка за период: 129378775.5


## 3. Выбор нужных столбцов и задание типов данных

### 3.1. Параметры usecols и dtype

Для экономии памяти можно читать только необходимые столбцы и сразу задавать подходящие типы данных.

In [3]:
csv_path = "../shared/data/orders_2024_02.csv"

use_columns = ["order_id", "user_id", "order_datetime", "price", "quantity"]
dtypes = {
    "order_id": "int64",
    "user_id": "int32",
    "price": "float32",
    "quantity": "int16"
}
parse_dates = ["order_datetime"]

df_orders = pd.read_csv(
    csv_path,
    usecols=use_columns,
    dtype=dtypes,
    parse_dates=parse_dates
)

print(df_orders.info())
print(df_orders.head())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        5000 non-null   int64         
 1   user_id         5000 non-null   int32         
 2   order_datetime  5000 non-null   datetime64[us]
 3   price           5000 non-null   float32       
 4   quantity        5000 non-null   int16         
dtypes: datetime64[us](1), float32(1), int16(1), int32(1), int64(1)
memory usage: 127.1 KB
None
   order_id  user_id      order_datetime         price  quantity
0      5001     1062 2024-02-17 11:39:03  16067.849609         1
1      5002     1078 2024-02-18 15:42:35   3716.949951         2
2      5003     1006 2024-02-21 15:12:54   9253.269531         1
3      5004     1094 2024-02-04 21:35:01  10408.889648         4
4      5005     1066 2024-02-20 00:56:21  14898.200195         2


### 3.2. Сравнение памяти с и без оптимизации

In [4]:
# Без оптимизации
df_full = pd.read_csv(csv_path)
print("Размер без оптимизации:")
print(df_full.info(memory_usage="deep"))

# С оптимизацией
df_opt = pd.read_csv(
    csv_path,
    usecols=use_columns,
    dtype=dtypes,
    parse_dates=parse_dates
)
print("Размер с оптимизацией:")
print(df_opt.info(memory_usage="deep"))

Размер без оптимизации:
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        5000 non-null   int64  
 1   user_id         5000 non-null   int64  
 2   order_datetime  5000 non-null   str    
 3   product_id      5000 non-null   int64  
 4   price           5000 non-null   float64
 5   quantity        5000 non-null   int64  
 6   status          5000 non-null   str    
dtypes: float64(1), int64(4), str(2)
memory usage: 390.6 KB
None
Размер с оптимизацией:
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        5000 non-null   int64         
 1   user_id         5000 non-null   int32         
 2   order_datetime  5000 non-null   datetime64[us]
 3   price           5000 non-nul

## 4. Импорт JSON (records и lines)

### 4.1. Формат JSON records

Файл `events_records.json` — JSON-список, где каждый элемент — словарь (формат records).

In [5]:
events_records_path = "../shared/data/events_records.json"

df_events_records = pd.read_json(events_records_path, orient="records")
print(df_events_records.head())
print(df_events_records.info())

   event_id  user_id          event_time   event_type         page
0         1     1095 2024-02-07 12:08:14  add_to_cart  /product/28
1         2     1058 2024-02-16 12:59:13     purchase   /product/7
2         3     1006 2024-01-19 00:14:21        click  /product/27
3         4     1092 2024-01-01 07:34:55       scroll  /product/26
4         5     1014 2024-02-17 19:46:47    page_view  /product/14
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   event_id    2000 non-null   int64         
 1   user_id     2000 non-null   int64         
 2   event_time  2000 non-null   datetime64[us]
 3   event_type  2000 non-null   str           
 4   page        2000 non-null   str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 113.6 KB
None


### 4.2. Формат JSON Lines (NDJSON)

Файл `events_lines.json` содержит по одному JSON-объекту в строке.

In [6]:
events_lines_path = "../shared/data/events_lines.json"

df_events_lines = pd.read_json(events_lines_path, lines=True)
print(df_events_lines.head())
print(df_events_lines.info())

   event_id  user_id          event_time   event_type         page
0      2001     1029 2024-03-19 00:34:38     purchase  /product/20
1      2002     1056 2024-03-25 17:16:08  add_to_cart        /cart
2      2003     1081 2024-02-06 06:09:49  add_to_cart  /product/30
3      2004     1005 2024-03-20 04:35:23     purchase   /product/5
4      2005     1033 2024-02-21 09:51:53        click  /product/11
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   event_id    2000 non-null   int64         
 1   user_id     2000 non-null   int64         
 2   event_time  2000 non-null   datetime64[us]
 3   event_type  2000 non-null   str           
 4   page        2000 non-null   str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 111.0 KB
None


## 5. Чтение Excel и работа с несколькими листами

Файл `products.xlsx` содержит два листа: `products` — список товаров, `categories` — список категорий.

In [7]:
excel_path = "../shared/data/products.xlsx"

df_products = pd.read_excel(excel_path, sheet_name="products")
print(df_products.head())
print(df_products.info())

   product_id product_name  category_id category_name  weight_kg
0           1    Product 1            1   Electronics      11.55
1           2    Product 2            3         Sport       2.22
2           3    Product 3            5          Toys      11.43
3           4    Product 4            1   Electronics       4.17
4           5    Product 5            5          Toys       1.81
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     30 non-null     int64  
 1   product_name   30 non-null     str    
 2   category_id    30 non-null     int64  
 3   category_name  30 non-null     str    
 4   weight_kg      30 non-null     float64
dtypes: float64(1), int64(2), str(2)
memory usage: 1.8 KB
None


Чтение нескольких листов сразу:

In [8]:
sheets = pd.read_excel(excel_path, sheet_name=["products", "categories"])

df_products = sheets["products"]
df_categories = sheets["categories"]

print(df_products.head())
print(df_categories.head())

   product_id product_name  category_id category_name  weight_kg
0           1    Product 1            1   Electronics      11.55
1           2    Product 2            3         Sport       2.22
2           3    Product 3            5          Toys      11.43
3           4    Product 4            1   Electronics       4.17
4           5    Product 5            5          Toys       1.81
   category_id category_name                  description
0            1   Electronics  Description for Electronics
1            2          Home         Description for Home
2            3         Sport        Description for Sport
3            4       Fashion      Description for Fashion
4            5          Toys         Description for Toys


## 6. Объединение нескольких файлов в один DataFrame

### 6.1. Объединение CSV-файлов по вертикали (concat)

In [9]:
files_pattern = "../shared/data/orders_2024_*.csv"
csv_files = glob.glob(files_pattern)

df_list = []
for file in csv_files:
    df_part = pd.read_csv(
        file,
        usecols=use_columns,
        dtype=dtypes,
        parse_dates=parse_dates
    )
    df_list.append(df_part)

df_orders_all = pd.concat(df_list, ignore_index=True)
print("Общая форма таблицы заказов:", df_orders_all.shape)

Общая форма таблицы заказов: (15000, 5)


### 6.2. Объединение заказов с товарами (merge)

In [10]:
# Чтобы объединить с товарами, нужно прочитать заказы с колонкой product_id
df_orders_full = pd.concat(
    [pd.read_csv(f, parse_dates=["order_datetime"]) for f in csv_files],
    ignore_index=True
)

df_orders_products = df_orders_full.merge(
    df_products,
    how="left",
    left_on="product_id",
    right_on="product_id"
)

print(df_orders_products.head())
print(df_orders_products.info())

   order_id  user_id      order_datetime  product_id     price  quantity  \
0         1     1044 2024-01-03 18:39:26          26  14038.15         1   
1         2     1079 2024-01-03 17:45:43          16   9172.60         1   
2         3     1041 2024-01-06 22:46:38          25   9035.26         2   
3         4     1083 2024-01-08 21:03:51           9  15234.33         3   
4         5     1068 2024-01-03 23:26:53          24   4134.38         2   

     status product_name  category_id category_name  weight_kg  
0  refunded   Product 26            3         Sport       0.74  
1      paid   Product 16            3         Sport       1.29  
2      paid   Product 25            3         Sport       0.20  
3      paid    Product 9            5          Toys       3.94  
4      paid   Product 24            1   Electronics       0.56  
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---

### 6.3. Объединение событий и заказов по user_id

In [11]:
df_user_orders = df_orders_full[["user_id", "order_id", "order_datetime"]]
df_user_events = df_events_lines[["user_id", "event_time", "event_type"]]

# пример: объединение по user_id (многие-ко-многим)
df_joined = df_user_events.merge(
    df_user_orders,
    on="user_id",
    how="inner"
)

print(df_joined.head())
print(df_joined.info())

   user_id          event_time event_type  order_id      order_datetime
0     1029 2024-03-19 00:34:38   purchase        77 2024-01-26 04:00:54
1     1029 2024-03-19 00:34:38   purchase        94 2024-01-03 19:03:04
2     1029 2024-03-19 00:34:38   purchase       300 2024-01-07 09:27:14
3     1029 2024-03-19 00:34:38   purchase       536 2024-01-07 23:10:39
4     1029 2024-03-19 00:34:38   purchase       591 2024-01-28 13:17:20
<class 'pandas.DataFrame'>
RangeIndex: 299890 entries, 0 to 299889
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   user_id         299890 non-null  int64         
 1   event_time      299890 non-null  datetime64[us]
 2   event_type      299890 non-null  str           
 3   order_id        299890 non-null  int64         
 4   order_datetime  299890 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(2), str(1)
memory usage: 13.7 MB
None
